# Lesson 3 · Phase 3 — Agentic Patterns: ReAct & Prompt Chaining

**Mastering Agentic AI Certification · Pre-read**

> Reasoning (Phase 2) happens *inside the model's head*. To actually **do things** — call tools, fetch data, take multi-step actions — we need patterns that let the model **act on the world** and **compose steps**. These are the prompting patterns that turn a model into an **agent**.

## 1. The complete picture — reason + act, and pipelines

```
        ReAct  (one model, a loop with tools)            PROMPT CHAINING (a pipeline of calls)

   ┌──────────────────────────────────┐            prompt A ─▶ output A
   │ Thought: I need today's FX rate   │                         │
   │ Action: search("USD INR")         │ ── tool ──▶             ▼
   │ Observation: 83.2                 │ ◀─ result ─    prompt B (uses A) ─▶ output B
   │ Thought: now multiply...          │                         │
   │ Action: calc(100*83.2)            │                         ▼
   │ Observation: 8320                 │                  prompt C (uses B) ─▶ FINAL
   │ Answer: ₹8320                     │
   └──────────────────────────────────┘
        loop until done                              fixed, predetermined steps
```

**ReAct** = dynamic loop (the model decides each next action). **Prompt chaining** = a fixed pipeline you design. Real agents use both.

## 2. ReAct (Reason + Act)

The model alternates **Thought → Action → Observation** until it can answer:

- **Thought:** reason about what to do next (this is CoT, reused).
- **Action:** call a **tool** (search, calculator, code run, API).
- **Observation:** the tool's result is fed back into the prompt.
- Loop until the model emits a final **Answer**.

```
Question: What is 100 USD in INR if the rate is 83.2?
Thought: I should multiply 100 by the rate.
Action: calculator("100 * 83.2")
Observation: 8320
Thought: That's the converted amount.
Answer: 8320 INR
```

This is the **core pattern behind tool-using agents** — grounding the model's reasoning in real, fresh data instead of its frozen knowledge.

In [ ]:
# ReAct loop demo (simulated): a model that REASONS, calls TOOLS, reads OBSERVATIONS.
# The 'policy' here is hand-coded to mimic what a real LLM would emit each turn.

# --- Tools the agent can call ---
def calculator(expr):
    return eval(expr, {"__builtins__": {}}, {})   # toy; never eval untrusted input for real
FX = {"USD_INR": 83.2}
def search(query):
    return FX.get(query.replace(" ", "_").upper().replace("USD_INR","USD_INR"), "not found")

TOOLS = {"calculator": calculator, "search": search}

# --- A scripted policy standing in for the LLM's Thought/Action decisions ---
def react(question):
    transcript, answer = [], None
    plan = [("Thought", "I need the USD->INR rate, then multiply by 100."),
            ("Action",  ("search", "USD INR")),
            ("Thought", "Got the rate; now compute 100 * rate."),
            ("Action",  ("calculator", "100 * 83.2")),
            ("Answer",  "100 USD = {last} INR")]
    last = None
    for kind, payload in plan:
        if kind == "Action":
            tool, arg = payload
            last = TOOLS[tool](arg)
            transcript.append(f"Action: {tool}({arg!r})")
            transcript.append(f"Observation: {last}")
        elif kind == "Answer":
            answer = payload.format(last=last)
            transcript.append(f"Answer: {answer}")
        else:
            transcript.append(f"Thought: {payload}")
    return transcript, answer

trace, ans = react("What is 100 USD in INR?")
print("\n".join(trace))
print("\nFINAL:", ans)

## 3. Prompt Chaining

Break a complex task into a **pipeline of simpler prompts**, where each step's **output becomes the next step's input**. Each prompt does one thing well.

```
raw article ─▶ [extract key facts] ─▶ facts ─▶ [summarise] ─▶ summary ─▶ [translate to Hindi] ─▶ final
```

- ✅ Easier to design, debug, and validate each step; reduces error per call.
- ✅ You can swap models per step (cheap model for extraction, strong model for synthesis).
- ⚠️ Errors can **propagate** down the chain; latency adds up.
- Contrast with ReAct: chaining is a **fixed** route you design; ReAct **decides** the route at runtime.

In [ ]:
# PROMPT CHAINING demo (simulated): each stage transforms the previous output.
def stage_extract(text):
    facts = [s.strip() for s in text.split(".") if s.strip()]
    return facts[:3]                                  # pretend the model pulled key facts

def stage_summarise(facts):
    return "Summary: " + "; ".join(facts) + "."

def stage_translate(summary):
    table = {"server": "सर्वर", "Jupyter": "जुपिटर", "token": "टोकन"}
    out = summary
    for en, hi in table.items():
        out = out.replace(en, hi)
    return out.replace("Summary:", "सारांश:")

def chain(text, *stages):
    print("INPUT:", text, "\n")
    for fn in stages:
        text = fn(text)
        print(f"after {fn.__name__:18}: {text}")
    return text

article = ("The Jupyter server runs on port 8888. "
           "The token is empty so no login is needed. "
           "Anyone with the URL can run code.")
final = chain(article, stage_extract, stage_summarise, stage_translate)
print("\nFINAL OUTPUT:", final)

## 4. ReAct vs Prompt Chaining vs single prompt

| Pattern | Control flow | Tools? | Best for |
|---|---|---|---|
| **Single prompt** | one call | no | simple, self-contained tasks |
| **Prompt chaining** | **fixed** pipeline you design | optional per step | known multi-stage workflows (extract→summarise→translate) |
| **ReAct** | **dynamic** loop the model drives | yes (core) | open-ended tasks needing live data / tools / backtracking |

Real agent frameworks combine them: a **chain** of stages, where some stages are **ReAct loops** with tools.

## 5. How this contributes to agentic AI — the top of the stack

This is lifecycle step **[4]** from Lesson 1 — the **agentic layer** built on a pre-trained, fine-tuned, aligned model:

```
 [1] pre-train ─▶ [2] fine-tune ─▶ [3] align ─▶ [4] AGENTIC USE
                                                  ├─ Foundations (Phase 1): role, examples, format
                                                  ├─ Reasoning  (Phase 2): plan via CoT / ToT
                                                  └─ Action     (Phase 3): ReAct loops + prompt chains
```

- **ReAct** gives an agent **hands** — the ability to use tools, read results, and ground itself in reality (overcoming the frozen knowledge cutoff from Lesson 1).
- **Prompt chaining** gives an agent **structure** — decomposing big goals into reliable steps.
- Together with memory and planning, these patterns *are* what makes "agentic AI" agentic.

## 6. Key takeaways

1. **ReAct** = a **Thought → Action → Observation** loop; the model **uses tools** and reacts to results — the core of tool-using agents.
2. **Prompt chaining** = a **fixed pipeline** of prompts where each output feeds the next; easy to debug, but errors propagate.
3. ReAct's route is **decided at runtime**; a chain's route is **designed in advance** — real agents combine both.
4. These patterns ground a frozen model in **live data and actions**, overcoming its knowledge cutoff.
5. This is the **agentic layer [4]** — the payoff of the whole Pre-training → Fine-tuning → Alignment → Agents journey.

---
*This completes Lesson 3.* You can now shape a frozen model with **foundations → reasoning → action**, the full prompt-engineering toolkit for building agents.

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0]); print("Platform:", platform.platform())
print("Lesson 3 · Phase 3 (ReAct, Prompt Chaining) notebook is running ✓")